In [ ]:
# Q1: Select the healthcare assistant based on roll number parity
# Odd roll number: Diabetes Care Information Assistant
# Even roll number: Heart Health Information Assistant

ASSISTANT_TYPE = "Diabetes"  # Change to "Heart" for even roll numbers


In [ ]:
# Q2: Create a labelled corpus containing 40+ queries across 5 intent categories

if ASSISTANT_TYPE.lower() == "diabetes":
    intents = {
        "blood_glucose": [
            "What is a normal blood sugar reading?", "How do I check my glucose level?",
            "When should blood glucose be tested?", "What does fasting glucose mean?",
            "Why is my blood sugar high in the morning?", "How can I record glucose readings?",
            "What is a post meal glucose test?", "Explain blood glucose monitoring."
        ],
        "symptoms": [
            "What are common symptoms of diabetes?", "Why am I feeling very thirsty?",
            "Can diabetes cause frequent urination?", "What are signs of low blood sugar?",
            "Why do I feel tired all the time?", "Can blurry vision be related to diabetes?",
            "What symptoms should I watch for?", "I feel shaky and sweaty, what could it mean?"
        ],
        "nutrition": [
            "What foods can help manage blood sugar?", "Can I eat fruit with diabetes?",
            "What is a balanced meal for diabetes?", "Are sugary drinks bad for blood sugar?",
            "How can I plan healthy meals?", "Does fiber help glucose levels?",
            "What are healthier snack options?", "How should I read food labels?"
        ],
        "medication": [
            "What is insulin used for?", "How should I discuss medication with my doctor?",
            "What are diabetes medicines?", "Why is medication adherence important?",
            "Can I ask my doctor about side effects?", "What is metformin commonly prescribed for?",
            "How do I keep track of my medicines?", "What should I ask at a medication review?"
        ],
        "prevention_lifestyle": [
            "How can exercise help diabetes?", "What lifestyle changes reduce diabetes risk?",
            "How important is sleep for blood sugar?", "Can walking help glucose management?",
            "How can I build a healthy routine?", "Why is weight management discussed in diabetes?",
            "How often should I exercise?", "What habits support long term health?"
        ]
    }
else:
    intents = {
        "heart_symptoms": [
            "What are common signs of heart disease?", "Why do I feel chest discomfort?",
            "What symptoms can heart problems cause?", "Can heart disease cause fatigue?",
            "What are warning signs of a heart condition?", "Why do I feel breathless?",
            "What is heart palpitations?", "How can I recognize heart symptoms?"
        ],
        "blood_pressure": [
            "What is blood pressure?", "How do I measure blood pressure at home?",
            "What does high blood pressure mean?", "What is systolic pressure?",
            "What is diastolic pressure?", "How can I record blood pressure readings?",
            "Why should blood pressure be monitored?", "What affects a blood pressure reading?"
        ],
        "nutrition": [
            "What foods are good for heart health?", "How can I reduce salt in my diet?",
            "What is a heart healthy meal?", "Are vegetables good for the heart?",
            "How can I eat less saturated fat?", "What foods contain healthy fats?",
            "How do I plan a balanced diet?", "What are heart healthy snacks?"
        ],
        "medication": [
            "What are common heart medicines?", "Why are prescribed heart medicines important?",
            "What should I ask about medicine side effects?", "What is a statin used for?",
            "How can I keep track of my medicines?", "What is medication adherence?",
            "Can I discuss my prescription with a doctor?", "What should I ask at a medication review?"
        ],
        "prevention_lifestyle": [
            "How does exercise support heart health?", "How can I lower heart disease risk?",
            "Why is sleep important for the heart?", "Can walking improve cardiovascular health?",
            "How can stress affect heart health?", "What lifestyle habits help my heart?",
            "How can I quit smoking for heart health?", "Why is regular checkup important?"
        ]
    }

texts, labels = [], []
for intent, queries in intents.items():
    texts.extend(queries)
    labels.extend([intent] * len(queries))

print("Total queries:", len(texts))
print("Intent categories:", list(intents.keys()))


In [ ]:
# Q3: Create hierarchical taxonomy with 4+ major categories and 2 subcategories each

taxonomy = {
    "Monitoring": ["Measurements", "Tracking"],
    "Symptoms": ["Common symptoms", "Warning signs"],
    "Lifestyle": ["Nutrition", "Physical activity"],
    "Treatment": ["Medication", "Clinical consultation"]
}
print(taxonomy)


In [ ]:
# Q4: Identify domain concepts and relationships

concepts = [
    "Patient", "Blood Glucose" if ASSISTANT_TYPE.lower()=="diabetes" else "Blood Pressure",
    "Symptoms", "Nutrition", "Exercise", "Medication", "Healthcare Provider", "Monitoring", "Risk Factor"
]

relationships = [
    ("Patient", "experiences", "Symptoms"),
    ("Patient", "records", "Monitoring"),
    ("Monitoring", "measures", concepts[1]),
    ("Nutrition", "influences", concepts[1]),
    ("Exercise", "supports", "Health"),
    ("Medication", "may manage", "Health Condition"),
    ("Patient", "consults", "Healthcare Provider"),
    ("Healthcare Provider", "advises", "Patient"),
    ("Risk Factor", "may increase", "Health Condition"),
    ("Symptoms", "may indicate", "Health Condition"),
    ("Patient", "follows", "Nutrition"),
    ("Patient", "practices", "Exercise")
]
print("Concepts:", concepts)
print("Relationships:")
for rel in relationships:
    print(rel)


In [ ]:
# Q5: Text preprocessing - lowercase, tokenization, stopword removal and lemmatization

import re
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    tokens = re.findall(r'\\b[a-z]+\\b', text)
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

processed_texts = [preprocess(t) for t in texts]
print("Original:", texts[0])
print("Processed:", processed_texts[0])


In [ ]:
# Q6: TF-IDF feature extraction and classifier training

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    processed_texts, labels, test_size=0.25, random_state=42, stratify=labels
)

model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', LogisticRegression(max_iter=1000))
])
model.fit(X_train, y_train)
print("Model trained!")


In [ ]:
# Q7: Evaluate using accuracy, precision, recall, F1-score and confusion matrix

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=list(intents.keys()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(intents.keys()))
disp.plot(xticks_rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Q8: Domain-specific knowledge base

if ASSISTANT_TYPE.lower() == "diabetes":
    knowledge_base = {
        "blood_glucose": "Blood glucose monitoring helps track glucose patterns. Follow the measurement schedule recommended by your healthcare provider.",
        "symptoms": "Symptoms can vary. Seek professional evaluation for concerning, persistent, or worsening symptoms.",
        "nutrition": "Balanced meals, vegetables, fiber-rich foods and limiting sugary drinks may support glucose management. Individual needs vary.",
        "medication": "Use medication only as prescribed. Discuss side effects, missed doses and changes with your healthcare provider.",
        "prevention_lifestyle": "Regular physical activity, sleep, healthy habits and medical follow-up can support diabetes care."
    }
else:
    knowledge_base = {
        "heart_symptoms": "Heart symptoms may include chest discomfort, shortness of breath or unusual fatigue. Symptoms vary and should be assessed clinically.",
        "blood_pressure": "Blood pressure monitoring records systolic and diastolic readings. Use a validated device and follow your clinician's guidance.",
        "nutrition": "A heart-supportive eating pattern often includes vegetables, fruits, whole grains and limiting excess sodium and saturated fat.",
        "medication": "Use prescribed medication as directed and discuss side effects or changes with your healthcare provider.",
        "prevention_lifestyle": "Physical activity, sleep, avoiding smoking, managing stress and checkups may support cardiovascular health."
    }
print(knowledge_base)


In [ ]:
# Q9: Implement reasoning rules, emergency escalation, low confidence and unsupported query handling

emergency_terms = [
    "chest pain", "cannot breathe", "can't breathe", "severe shortness of breath",
    "fainting", "unconscious", "stroke", "severe bleeding"
]

def assistant_response(query):
    q = query.lower()
    # Rule 1: Emergency escalation takes priority
    if any(term in q for term in emergency_terms):
        return ("EMERGENCY: If this may be a medical emergency, contact local emergency services "
                "immediately. Do not rely on this assistant for emergency care.")

    # Rule 2: Classify query with model
    cleaned = preprocess(query)
    probabilities = model.predict_proba([cleaned])[0]
    confidence = max(probabilities)
    predicted_intent = model.classes_[probabilities.argmax()]

    # Rule 3: Low confidence handling
    if confidence < 0.30:
        return "I'm not confident I understood. Please rephrase or consult a healthcare professional."

    # Rule 4: Domain supported response retrieval
    if predicted_intent in knowledge_base:
        return knowledge_base[predicted_intent]

    # Rule 5: Unsupported-query handling
    return "I can only provide general information within this assistant's health topics. Please consult a healthcare professional."

print("Rules added.")


In [ ]:
# Q10: Integrate NLP, classification, reasoning and retrieval; test required cases

test_queries = [
    list(intents.values())[0][0],       # Intent query
    "Could you explain this in another way?",  # differently worded / may be low confidence
    "Tell me about a rare tropical infection.", # unsupported
    "xyz qwerty blah",                  # low confidence
    "I have severe chest pain and cannot breathe." # emergency
]

for q in test_queries:
    print("\\nQuery:", q)
    print("Response:", assistant_response(q))


In [ ]:
# Q11: Interactive assistant interface (Google Colab)

# In Colab, run this cell and use the input box to enter multiple queries.
# Stop by entering 'exit'.

try:
    while True:
        user_query = input("Ask your health assistant (or type exit): ")
        if user_query.strip().lower() == "exit":
            break
        print("Assistant:", assistant_response(user_query))
except EOFError:
    print("Interactive input is unavailable in this environment.")


In [ ]:
# Optional: Colab notebook can use ipywidgets for an interface
# Uncomment and run in Colab if desired.

# import ipywidgets as widgets
# from IPython.display import display
# query_box = widgets.Text(placeholder='Enter a healthcare question')
# output_box = widgets.Output()
# button = widgets.Button(description='Ask')
# def on_click(b):
#     with output_box:
#         output_box.clear_output()
#         print(assistant_response(query_box.value))
# button.on_click(on_click)
# display(query_box, button, output_box)
